# L2. Kinematyka i sterowanie kinematyczne manipulatora w UAIbot

**Systemy mechatroniczne · Laboratorium 2 (3 h akad.)** · wykłady W04, W05, W06

[![Otwórz w Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bpasz/pwr-systemy-mechatroniczne-dla-studenta/blob/main/laboratoria/L2-uaibot-kinematyka/L2_uaibot.ipynb)

Notatnik bazuje na otwartym kursie **UAIbot: Robotic Manipulators** (V. M. Gonçalves i in.) i bibliotece `uaibot` (Python).
Symulacje 3D wyświetlają się w przeglądarce: obracaj widok myszą, a na dole okna jest odtwarzacz animacji.

**Plan:** część A (prowadzona): MTJ, kinematyka prosta z DH, Jakobian i osobliwości.
Część B (mini-projekt): sterowanie kinematyczne pozą i śledzenie okręgu.

In [ ]:
%pip install -q uaibot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import uaibot as ub

np.set_printoptions(precision=4, suppress=True)

## A1. Macierze transformacji jednorodnej

Złożenie z wykładu W04: $H = R_x(\pi/4)\,D_y(-0{,}25)\,R_z(\pi/2)\,D_x(1)$.
W UAIbot obroty i przesunięcia to od razu macierze 4×4: `Utils.rotx`, `Utils.roty`, `Utils.rotz`, `Utils.trn`.

In [ ]:
H = ub.Utils.rotx(np.pi/4) * ub.Utils.trn([0, -0.25, 0]) * ub.Utils.rotz(np.pi/2) * ub.Utils.trn([1, 0, 0])
print(H)

Q, s = H[0:3, 0:3], H[0:3, 3]
print("Q^T Q = I ?", np.allclose(Q.T * Q, np.eye(3)), "  det(Q) =", round(np.linalg.det(Q), 6))

F0 = ub.Frame(htm=np.identity(4), name="F0", size=0.4)
F1 = ub.Frame(htm=H, name="F1", size=0.4)
sim = ub.Simulation([F0, F1])
sim.run()

**Pytania A1** (odpowiedz w sprawozdaniu):

1. Odczytaj z wydruku osie $x_1, y_1, z_1$ i środek układu $\mathcal{F}_1$ zapisane w $\mathcal{F}_0$. Sprawdź na symulacji.
2. Policz $R_x(90°)R_y(90°)$ i $R_y(90°)R_x(90°)$ (kod poniżej). Jaka jest oś i kąt każdego obrotu (`Utils.axis_angle`)? Porównaj z W04.

In [ ]:
A = ub.Utils.rotx(np.pi/2) * ub.Utils.roty(np.pi/2)
B = ub.Utils.roty(np.pi/2) * ub.Utils.rotx(np.pi/2)
for nazwa, M in [("Rx Ry", A), ("Ry Rx", B)]:
    os_obrotu, kat = ub.Utils.axis_angle(M)
    print(nazwa, " oś:", np.round(os_obrotu.T, 4), " kąt [°]:", round(np.degrees(kat), 2))

## A2. Kinematyka prosta KUKA KR5 z parametrów DH

Każde ogniwo przekształca układ $\mathcal{F}_{DH(i-1)}$ w $\mathcal{F}_{DHi}$:
$H_{DH(i-1)}^{DHi} = R_z(\theta_i)\,D_z(d_i)\,R_x(\alpha_i)\,D_x(a_i)$, a dla przegubu obrotowego $\theta_i$ zawiera $q_i$.

In [ ]:
robot = ub.Robot.create_kuka_kr5()
print(" i  typ   theta      d     alpha      a")
for i, o in enumerate(robot.links):
    print(f"{i+1:2d}  {'R' if o.joint_type == 0 else 'P'}   {o.theta:6.3f} {o.d:7.3f} {o.alpha:7.3f} {o.a:7.3f}")

In [ ]:
def mtj_dh(theta, d, alpha, a):
    """MTJ przejścia między kolejnymi układami DH."""
    return ub.Utils.rotz(theta) * ub.Utils.trn([0, 0, d]) * ub.Utils.rotx(alpha) * ub.Utils.trn([a, 0, 0])


def kinematyka_prosta(robot, q):
    """Własna kinematyka prosta: łańcuch MTJ od podstawy do efektora."""
    H = robot.htm_base_0                       # podstawa -> F_DH0
    for i, o in enumerate(robot.links):
        H = H * mtj_dh(o.theta + q[i, 0], o.d, o.alpha, o.a)
    return H * robot.htm_n_eef                 # F_DHn -> efektor


q = np.matrix([0.3, -0.5, 0.8, 0.1, 0.4, -0.2]).T
H_moja = kinematyka_prosta(robot, q)
H_uaibot = robot.fkm(q=q)
print(H_moja)
print("maks. różnica względem robot.fkm():", np.abs(H_moja - H_uaibot).max())

**Zadanie A2.** Dla 5 losowych konfiguracji sprawdź, że Twoja funkcja zgadza się z `robot.fkm()` (kod poniżej).
Następnie wyświetl robota w konfiguracji `q` razem z układem `ub.Frame(htm=H_moja)`: układ musi „siedzieć” na kołnierzu robota.

In [ ]:
rng = np.random.default_rng(0)
for _ in range(5):
    q_los = np.matrix(rng.uniform(-np.pi, np.pi, 6)).T
    print(np.abs(kinematyka_prosta(robot, q_los) - robot.fkm(q=q_los)).max())

robot.add_ani_frame(time=0, q=q)
sim = ub.Simulation([robot, ub.Frame(htm=H_moja, size=0.2)])
sim.run()

## A3. Jakobian geometryczny i osobliwości

`robot.jac_geo(q)` zwraca Jakobian $6\times n$ oraz MTJ efektora. Sprawdzimy górne 3 wiersze ($J_v$) różnicami centralnymi:
$\dfrac{\partial s_e}{\partial q_j} \approx \dfrac{s_e(q + h e_j) - s_e(q - h e_j)}{2h}$.

In [ ]:
Jg, _ = robot.jac_geo(q=q)
h = 1e-3
Jv_num = np.zeros((3, 6))
for j in range(6):
    dq = np.matrix(np.zeros((6, 1)))
    dq[j] = h
    Jv_num[:, j] = ((robot.fkm(q=q + dq)[0:3, 3] - robot.fkm(q=q - dq)[0:3, 3]) / (2*h)).A1
print("maks. różnica J_v:", np.abs(Jv_num - Jg[0:3, :]).max())

for nazwa, qq in [("q0 (konfiguracja początkowa)", robot.q0), ("q", q)]:
    J, _ = robot.jac_geo(q=qq)
    sv = np.linalg.svd(J, compute_uv=False)
    print(f"{nazwa}: wartości szczególne {np.round(sv, 4)}, rząd = {np.linalg.matrix_rank(J, tol=1e-6)}")

**Pytania A3:**

1. Dlaczego konfiguracja początkowa KUKA KR5 jest osobliwa (patrz W06, animacja z UAIbot)? Który ruch efektora jest w niej niemożliwy?
2. Oblicz $\dot{q} = J^{\dagger(\epsilon)}\xi_d$ dla $\xi_d = (0{,}1\ 0\ 0\ 0\ 0\ 0)^T$ w `q0` i w `q`, dla $\epsilon = 10^{-6}$ i $10^{-2}$ (`ub.Utils.dp_inv`). Porównaj normy $\dot{q}$ i skomentuj.

In [ ]:
xi_d = np.matrix([0.1, 0, 0, 0, 0, 0]).T
for nazwa, qq in [("q0", robot.q0), ("q", q)]:
    J, _ = robot.jac_geo(q=qq)
    for eps in [1e-6, 1e-2]:
        qdot = ub.Utils.dp_inv(J, eps) * xi_d
        blad = np.linalg.norm(J * qdot - xi_d)
        print(f"{nazwa}, eps={eps:g}: |qdot| = {np.linalg.norm(qdot):8.4f} rad/s, |J qdot - xi_d| = {blad:.2e}")

## B. Mini-projekt: sterowanie kinematyczne

Model robota: $\dot{q} = u$ (interfejs prędkości przegubów). Prawo sterowania z W06:

$$u = -J_r^{\dagger(\epsilon)}(q)\big(F(r) + \tfrac{\partial r}{\partial t}\big), \qquad F(r) = K r$$

Funkcja zadania dla pełnej pozy: $r = \big(s_e - s_d,\ 1 - x_d^Tx_e,\ 1 - y_d^Ty_e,\ 1 - z_d^Tz_e\big)$.
Funkcja `symuluj_poze` poniżej realizuje to prawo i zapisuje historię $r(t)$ i $u(t)$.

In [ ]:
def funkcja_zadania_pozy(robot, q, H_d):
    """Zwraca r (6x1) i Jakobian zadania Jr (6xn) dla pełnej pozy efektora."""
    x_d, y_d, z_d, s_d = H_d[0:3, 0], H_d[0:3, 1], H_d[0:3, 2], H_d[0:3, 3]
    Jg, H = robot.jac_geo(q=q)
    x_e, y_e, z_e, s_e = H[0:3, 0], H[0:3, 1], H[0:3, 2], H[0:3, 3]
    r = np.vstack([s_e - s_d, 1 - x_d.T*x_e, 1 - y_d.T*y_e, 1 - z_d.T*z_e])
    Jr = np.vstack([Jg[0:3, :],
                    x_d.T * ub.Utils.S(x_e) * Jg[3:6, :],
                    y_d.T * ub.Utils.S(y_e) * Jg[3:6, :],
                    z_d.T * ub.Utils.S(z_e) * Jg[3:6, :]])
    return r, Jr


def symuluj_poze(K=2.0, eps=1e-3, tmax=6.0, dt=0.01, F=None, animacja=False):
    """Sterowanie pozą KUKA KR5. F: funkcja r -> F(r); domyślnie F(r) = K r."""
    robot = ub.Robot.create_kuka_kr5()
    H_d = ub.Utils.trn([-0.3, 0.2, -0.3]) * robot.fkm()   # poza docelowa (jak w W06)
    F = F or (lambda r: K * r)
    t, hist_t, hist_r, hist_u = 0.0, [], [], []
    for _ in range(round(tmax / dt)):
        q = robot.q                                        # "pomiar" konfiguracji
        r, Jr = funkcja_zadania_pozy(robot, q, H_d)
        u = -ub.Utils.dp_inv(Jr, eps) * F(r)               # prawo sterowania
        robot.add_ani_frame(time=t + dt, q=q + u*dt)       # robot wykonuje u przez dt
        hist_t.append(t); hist_r.append(r.A1); hist_u.append(u.A1)
        t += dt
    if animacja:
        ub.Simulation([robot, ub.Frame(htm=H_d, size=0.2)]).run()
    return np.array(hist_t), np.array(hist_r), np.array(hist_u)


def wykresy(t, r, u, tytul=""):
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].plot(t, r); ax[0].set_title("funkcja zadania r(t) " + tytul); ax[0].set_xlabel("t [s]")
    ax[0].legend([f"r{i+1}" for i in range(r.shape[1])], fontsize=8)
    ax[1].plot(t, u); ax[1].set_title("sterowanie u(t) [rad/s]"); ax[1].set_xlabel("t [s]")
    ax[1].legend([f"u{i+1}" for i in range(u.shape[1])], fontsize=8)
    plt.tight_layout(); plt.show()


t, r, u = symuluj_poze(K=2.0, eps=1e-3, animacja=True)
wykresy(t, r, u, "(K = 2)")

### B1. Wpływ $K$ i $\epsilon$

Uzupełnij tabelę: czas zbieżności (pierwsza chwila, od której $\max_i |r_i| < 0{,}01$) i maksymalna prędkość przegubu $\max |u|$.
Wnioski odnieś do slajdu „Prawo sterowania” (W06): reguła $5/K$, kompromis prędkość–wykonalność.

In [ ]:
def czas_zbieznosci(t, r, prog=0.01):
    ok = np.max(np.abs(r), axis=1) < prog
    for i in range(len(ok)):
        if ok[i:].all():
            return t[i]
    return np.nan

print("   K      eps    t_zb [s]   max|u| [rad/s]")
for K in [0.5, 2.0, 8.0]:
    for eps in [1e-4, 1e-3, 1e-1]:
        t, r, u = symuluj_poze(K=K, eps=eps, tmax=12.0)
        print(f"{K:5.1f} {eps:8.0e} {czas_zbieznosci(t, r):9.2f} {np.abs(u).max():12.3f}")

### B2. Funkcja nasycenia $F(r)$

Z W06 (i UAIbot Class 5): zamiast $F(r) = Kr$ użyj funkcji o stałej szybkości zaniku z liniowym obszarem blisko zera:
$F_i(r) = A\,\mathrm{sat}(r_i / w_{tol})$, gdzie $\mathrm{sat}$ obcina do $[-1, 1]$. Parametry: $A = 0{,}25$, $w_{tol} = 0{,}01$.
Porównaj wykresy z przypadkiem $K = 2$: czy orientacja zbiega równomierniej?

In [ ]:
# TODO(B2): zdefiniuj F_nasycenie(r) i uruchom symulację, np.:
#   F_nasycenie = lambda r: A * np.clip(r / w_tol, -1, 1)
#   t, r, u = symuluj_poze(F=F_nasycenie, animacja=True); wykresy(t, r, u, "(nasycenie)")

### B3. Śledzenie okręgu z członem feedforward

Zadanie zmienne w czasie: środek efektora ma poruszać się po okręgu
$s_d(t) = (R\cos\omega t,\ y_c,\ z_c + R\sin\omega t)$, $R = 0{,}2$ m, $y_c = z_c = 0{,}6$ m, $\omega = \pi/2$ rad/s,
a oś $z$ efektora ma być równoległa do osi $y$ świata: $z_d = (0\ 1\ 0)^T$.

Funkcja zadania (4 składowe): $r = (s_e - s_d(t),\ 1 - z_d^Tz_e)$, Jakobian: $J_r = (J_v;\ z_d^TS(z_e)J_\omega)$,
feedforward: $\partial r/\partial t = (-\dot{s}_d(t);\ 0)$.

**Zadanie:** zaimplementuj sterownik, pokaż animację (dodaj `ub.PointCloud` z punktami okręgu) i wykres błędu położenia $\|s_e - s_d\|$.
Porównaj błąd **z** i **bez** członu feedforward.

In [ ]:
# TODO(B3): uzupełnij na wzór symuluj_poze().
R, y_c, z_c, w = 0.2, 0.6, 0.6, np.pi/2
s_d = lambda t: np.matrix([R*np.cos(w*t), y_c, z_c + R*np.sin(w*t)]).T
s_d_kropka = lambda t: np.matrix([-R*w*np.sin(w*t), 0, R*w*np.cos(w*t)]).T
z_d = np.matrix([0, 1, 0]).T

def symuluj_okrag(K=2.0, eps=1e-3, tmax=12.0, dt=0.01, feedforward=True, animacja=False):
    robot = ub.Robot.create_kuka_kr5()
    ...  # TODO: pętla sterowania, zwróć (t, bledy_polozenia)

## Sprawozdanie

Sprawozdanie piszesz na bieżąco w trakcie zajęć (katalog `L2/` w Twoim repozytorium: ten notatnik z wynikami, File → Download .ipynb, oraz `README.md`) i przed końcem zajęć przedstawiasz prowadzącemu do oceny. `README.md` zawiera:

- odpowiedziami na pytania A1–A3,
- tabelą z B1 i wnioskami (2–3 zdania),
- porównaniem B2 z przypadkiem $F(r) = Kr$ (wykresy),
- wykresem błędu z B3 z feedforward i bez, z interpretacją.

**Źródła:** UAIbot, *Robotic Manipulators*, Class 2–5 (<https://uaibot.github.io/class_presentations/>); dokumentacja UAIbotPy (<https://uaibot.github.io/>).